# Experiment Visualizations

### Imports

In [1]:
%matplotlib inline
%load_ext autoreload
%autoreload 1

import numpy as np
import matplotlib.pyplot as plt

# import matplotlib
# matplotlib.use("pgf")
# matplotlib.rcParams.update({
#     "pgf.texsystem": "pdflatex",
#     'font.family': 'serif',
#     'text.usetex': True,
#     'pgf.rcfonts': False,
# })

import seaborn as sns
sns.set(rc={'figure.figsize':(11,4)})
import pandas as pd
import re
from ipywidgets import interactive, interact, fixed, interact_manual, SelectMultiple
from IPython.display import display
import ipywidgets as widgets

### Globals

In [2]:
experiment_path = '../experiment_results/experiment'

### Experiment Description

In [3]:
experiment_description = {
    0: "",
    1: "",
    2: "MAIN EXPERIMENT",
    3: "",
    4: "",
    5: "",
    6: "MAIN EXPERIMENT",
    7: "same as 6 but only region where lines cross over",
    8: "Same as 7, bigger machine",
    9: "Constant size dataset",
    16: "[Server] n2-standard-8 (8 vCPUs, 32 GB memory)\n"
        "[Disk] 2.5 TB network SSD\n"
        "[Versions] TFRecord, RR, Arrow"
        "[Series_Param] Bytes per Row",
    21: "[Server] n2-standard-32 (32 vCPUs, 128 GB memory)\n"
        "[Disk] 2.5 TB network SSD\n"
        "[Versions] TFRecord, RR, Arrow"
        "[Series_Param] Bytes per Row",
    22: "[Server] n2-standard-32 (32 vCPUs, 128 GB memory)\n"
        "[Disk] 2.5 TB network SSD\n"
        "[Versions] TFRecord, RR, Arrow"
        "[Series_Param] Bytes per Row"
        "[Constant Dataset Size] = True",
    23: "Same as 16 with bigger machine (32 vCPUs etc.)",
    24: "Local experiment for time analysis TFRecord / Arrow",
    25: "Memory experiments 25 - 30. Wheel: 4a550a90",
    26: "Wheel: 4a550a90",
    27: "Wheel: 4a550a90",
    28: "Wheel: 4a550a90",
    29: "Wheel: 29726547",
    30: "Wheel: 29726547",
    35: "Custom bigger machine, 100GB dataset const size",
    36: "Exploring parallelism",
    99: "Custom experiment, varying."
}

### Plot iostats

In [4]:
def plot_iostat(exp_id, subplots):
    path = experiment_path + str(exp_id)
    # plt.show()
    iostat = pd.read_csv(path + '/iostat.csv', index_col=0)
    cols_mask = [False, True, True, False, False, False, False]
    iostat.loc[:,cols_mask].plot(marker='.', alpha=0.5, figsize=(16, 9), subplots=subplots)
    plt.ylabel("MB/s", fontsize=16)
    plt.xlabel("Timestamp", fontsize=16)
    plt.title("Disk I/O", fontsize=23)
    plt.xticks(rotation=15)
    plt.savefig('output_mem', dpi=600)
    plt.show()
    print(experiment_description[exp_id])
    return

interact(plot_iostat, exp_id=widgets.IntSlider(value=99, min=0, max=100), subplots=[True, False])

interactive(children=(IntSlider(value=99, description='exp_id'), Dropdown(description='subplots', options=(Tru…

<function __main__.plot_iostat(exp_id, subplots)>

### Plot Execution Time with increasing Bytes/Row

In [9]:
index_params = ['version', 'bytes/row', 'workers', 'annotated']
modes = ['write_time', 'read_time']
annotated = [True, False]
versions = ['TFRecord', 'RR', 'Arrow']
workers = ['Single', 'Parallel']
exp_ids = [2, 5, 6, 7, 8, 9, 16, 23, 35, 37, 99]

def keep_all(col_name, values, df):
    mask = []
    for s in df[col_name]:
        if s in values:
            mask.append(True)
        else:
            mask.append(False)
    return df.loc[mask]

def remove_all(col_name, values, df):
    mask = []
    for s in df[col_name]:
        if s in values:
            mask.append(False)
        else:
            mask.append(True)
    return df.loc[mask]

def get_col_title(version, workers, annotated, mode):
    return version[0] + '_p=' + str(workers) + ('T' if annotated else 'F') + mode[0]

def get_column(df, modes_diff):
    df_out = pd.DataFrame()
    for k in df['bytes/row']:
        df_out[str(k)] = df.loc[df['bytes/row'] == k, ['write_time', 'read_time']].squeeze()
    if modes_diff:
        df_out = df_out.drop(modes_diff)
    return df_out

def plot_bpr(exp_id, versions, workers, annotated, modes, logx, logy):
    path = experiment_path + str(exp_id)
    stats = pd.read_csv(path + '/stats.csv')
    stats = keep_all('version', versions, stats)
    if len(workers) == 1:
        if workers[0] == 'Single':
            stats = keep_all('workers', [1], stats)
        else:
            stats = remove_all('workers', [1], stats)
    stats = keep_all('annotated', annotated, stats)
    modes_diff = [x for x in ['write_time', 'read_time'] if x not in modes]
    stats = stats.drop(['datatype', 'columns', 'rows'], axis=1)
    try:
        stats = stats.drop(['rep_id'], axis=1)
    except:
        print('Drop not possible...')
    std_dev = stats.groupby(index_params, as_index=False).agg(np.std)
    stats = stats.groupby(index_params, as_index=False).mean()
    std_dev = std_dev.groupby([x for x in index_params if x != 'bytes/row']).apply(get_column, modes_diff)
    stats = stats.groupby([x for x in index_params if x != 'bytes/row']).apply(get_column, modes_diff)
    new_index = []
    for (v, p, a, m) in stats.index:
        new_index.append(get_col_title(v,p,a, m))
    stats.index = new_index
    std_dev.index = new_index
    stats = stats.transpose()
    std_dev = std_dev.transpose()
    plt.figure(figsize=(16,9))
    x = []
    for i in stats.index:
        x.append(int(i))
    x = np.asarray(x)
    if logx:
        x = np.log2(x)
    legend=[]
    for c in stats.columns:
        legend.append(c)
        y = stats[c].to_numpy()
        if logy:
            y = np.log2(y)
        e = std_dev[c].to_numpy()
        plt.errorbar(x, y, e, alpha=0.7, linewidth=2, linestyle='-', marker='.')
    plt.legend(legend, loc=2, prop={'size': 14})
    if logy:
        plt.ylabel("Log2 Time (s)", fontsize=16)
    else:
        plt.ylabel("Time (s)", fontsize=16)
    if logx:
        plt.xlabel("Log2 Bytes / Row", fontsize=16)
    else:
        plt.xlabel("Bytes / Row", fontsize=16)
    plt.title("Overall Execution Time", fontsize=23)
    plt.xticks(rotation=25)
    plt.savefig('output', dpi=600)
    plt.show()

interact(plot_bpr, exp_id=exp_ids, logx=[True, False], logy=[False, True],
         versions=SelectMultiple(options=versions),
         workers=SelectMultiple(options=workers),
         modes=SelectMultiple(options=modes),
         annotated=SelectMultiple(options=annotated))


interactive(children=(Dropdown(description='exp_id', options=(2, 5, 6, 7, 8, 9, 16, 23, 35, 37, 99), value=2),…

<function __main__.plot_bpr(exp_id, versions, workers, annotated, modes, logx, logy)>

### Plot Memory Leak

In [6]:
def plot_experiments_10_to_15(ids, subplots):
    # read mem_stat data from experiments
    print("memory")

    df = pd.DataFrame()
    for i in ids:
        df_temp = pd.DataFrame(columns=['mem_bytes'])
        file = open(experiment_path + str(i) + '/mem_log.txt')
        j = 0
        for line in file:
            if 'r' in line:
                continue

            line = re.sub(" +", " ", line)
            line = line.split(" ")
            active_mem = line[6]
            active_mem = float(active_mem) / 1000
            timestamp = line[-2] + " " + line[-1][:-1]
            timestamp = pd.to_datetime(timestamp)
            df_temp.loc[j] = (active_mem)
            j+=1
        df['mem_exp_' + str(i)] = df_temp['mem_bytes']
    df.plot(marker='.', alpha=0.5, figsize=(16, 9), subplots=subplots)
    plt.ylabel("MB in memory", fontsize=16)
    plt.xlabel("Time (s)", fontsize=16)
    plt.title("Active Memory over Time", fontsize=23)
    plt.xticks(rotation=25)
    plt.savefig('output', dpi=600)
    plt.show()

interact(plot_experiments_10_to_15, ids=SelectMultiple(options=[10, 11, 12, 13, 14, 15, 17, 18, 19, 20]), subplots=[False, True])

interactive(children=(SelectMultiple(description='ids', options=(10, 11, 12, 13, 14, 15, 17, 18, 19, 20), valu…

<function __main__.plot_experiments_10_to_15(ids, subplots)>

### Plot Memory Leak II

In [ ]:
exp_description = {

    25: "TFRecord, same dataset",
    26: "RR, same dataset",
    27: "TFRecord, vary bpr",
    28: "RR, vary bpr",
    29: "TFRecord, vary bpr, no stats",
    30: "RR, vary bpr, no stats",
    31: "TFRecord, vary bpr, not Annot., no stats",
    32: "RR, vary bpr, not Annot., no stats",
    33: "TFRecord, vary bpr, no stats, gc betw. w/r",
    34: "RR, vary bpr, no stats, gc betw. w/r"
}


def plot_experiments_25_to_30(ids, subplots):
    # read mem_stat data from experiments
    print("memory")

    df = pd.DataFrame()
    for i in ids:
        df_temp = pd.DataFrame(columns=['mem_bytes'])
        file = open(experiment_path + str(i) + '/mem_log.txt')
        j = 0
        for line in file:
            if 'r' in line:
                continue

            line = re.sub(" +", " ", line)
            line = line.split(" ")
            active_mem = line[6]
            active_mem = float(active_mem) / 1000
            timestamp = line[-2] + " " + line[-1][:-1]
            timestamp = pd.to_datetime(timestamp)
            df_temp.loc[j] = (active_mem)
            j+=1
        df['mem_exp_' + str(i)] = df_temp['mem_bytes']
    df.plot(marker='.', alpha=0.5, figsize=(16, 9), subplots=subplots)
    legend = []
    for id in ids:
        legend.append(exp_description[id])
    plt.legend(legend)
    plt.ylabel("MB in memory", fontsize=16)
    plt.xlabel("Time (s)", fontsize=16)
    plt.title("Active Memory over Time", fontsize=23)
    plt.xticks(rotation=25)
    # plt.savefig('memory.pgf')
    plt.savefig('output', dpi=300)
    plt.show()

interact(plot_experiments_25_to_30, ids=SelectMultiple(options=[25, 26, 27, 28, 29, 30, 31, 32, 33, 34]), subplots=[False, True])

interactive(children=(SelectMultiple(description='ids', options=(25, 26, 27, 28, 29, 30, 31, 32, 33, 34), valu…

<function __main__.plot_experiments_25_to_30(ids, subplots)>